# 02. Stage 1A: Content-Based Retrieval (BM25)
 
 Notebook này xây dựng tầng lọc thô dựa trên nội dung (Content-Based) để đề xuất các ứng viên ban đầu cho người dùng sử dụng thuật toán BM25.
 
 ---
 
 ### Phân tích Quyết định Thiết kế:
 *   **Tại sao chọn BM25 thay vì TF-IDF?**
     *   BM25 tích hợp hai cơ chế tiên tiến hơn TF-IDF truyền thống: **Bão hòa Tần suất từ (TF Saturation)** giúp giới hạn tầm ảnh hưởng của một từ khóa lặp lại quá nhiều lần, và **Chuẩn hóa Độ dài Tài liệu (Document Length Normalization)** giúp cân bằng điểm số giữa các phim có metadata ngắn gọn và phim có metadata dài dòng.
 *   **Tại sao không chọn Deep Learning (Sentence-BERT)?**
     *   Sentence-BERT (SBERT) là mạng Transformer dùng để hiểu **ngữ nghĩa tự nhiên** của các câu văn tự do (như `overview`). Với dữ liệu từ khóa rời rạc (như genres hay tên diễn viên), SBERT không mang lại lợi ích về ngữ nghĩa mà còn gây ra Overhead tính toán cực lớn (tải model ~400MB, suy luận chậm trên CPU). BM25 là đủ và hiệu quả hơn rất nhiều cho keyword matching.
 *   **Tại sao không chọn Word2Vec / FastText?**
     *   Các mô hình Word Embedding tĩnh yêu cầu khối lượng văn bản cực lớn để huấn luyện các mối quan hệ từ vựng, hoặc nếu dùng pre-trained thì thường không tối ưu cho các danh từ riêng (tên đạo diễn, diễn viên) hay thuật ngữ điện ảnh đặc thù.
 

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
 
# Thêm đường dẫn cha để import recsys_utils
sys.path.append(os.path.abspath('..'))
from recsys_utils import BM25
 
# Load phim
movies_df = pd.read_csv(os.path.join("..", "..", "data", "crawler", "movies_crawled.csv"))
 
# Xử lý missing values
movies_df['genres'] = movies_df['genres'].fillna('')
movies_df['director'] = movies_df['director'].fillna('')
movies_df['cast'] = movies_df['cast'].fillna('')
movies_df['keywords'] = movies_df['keywords'].fillna('')
 
# 1. Kết hợp đặc trưng dạng văn bản
def build_metadata_soup(row):
    genres = row['genres'].replace('|', ' ')
    cast = ' '.join(row['cast'].split('|')[:5])
    keywords = row['keywords'].replace('|', ' ')
    director = row['director'].replace(' ', '')
    return f"{genres} {director} {cast} {keywords}"
 
movies_df['soup'] = movies_df.apply(build_metadata_soup, axis=1)
display(movies_df[['title', 'soup']].head(3))
 

In [ ]:
# 2. Xây dựng mô hình BM25 và TF-IDF Matrix (TF-IDF dùng cho MMR)
bm25 = BM25()
bm25.fit(movies_df['soup'])
 
tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(movies_df['soup'])
 
print(f"BM25 fitted on {len(movies_df)} movies.")
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
 
# Lưu trữ ma trận tương đồng và vectorizer
os.makedirs("models", exist_ok=True)
with open("models/bm25_model.pkl", "wb") as f:
    pickle.dump(bm25, f)
    
with open("models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
     
with open("models/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)
 

In [ ]:
# 3. Định nghĩa hàm gợi ý Content-Based dùng BM25 cho một danh sách phim đã xem
def get_content_based_candidates(liked_movie_ids, top_n=100):
    liked_idx = movies_df[movies_df['movieId'].isin(liked_movie_ids)].index.tolist()
    if not liked_idx:
        return movies_df.sort_values(by='popularity', ascending=False)['movieId'].head(top_n).tolist()
        
    # Tạo query bằng cách gộp soup của các phim đã xem
    liked_soups = movies_df.iloc[liked_idx]['soup'].tolist()
    query = " ".join(liked_soups)
    
    scores = bm25.transform(query)
    sorted_idx = np.argsort(scores)[::-1]
    
    liked_idx_set = set(liked_idx)
    candidate_indices = [idx for idx in sorted_idx if idx not in liked_idx_set]
    
    recommended_movie_ids = movies_df.iloc[candidate_indices]['movieId'].head(top_n).tolist()
    return recommended_movie_ids

# Test thử nghiệm gợi ý
test_likes = [1339713, 1084244]
candidates = get_content_based_candidates(test_likes, top_n=5)
print("Phim đã xem:", movies_df[movies_df['movieId'].isin(test_likes)]['title'].tolist())
print("Gợi ý Content-based:", movies_df[movies_df['movieId'].isin(candidates)]['title'].tolist())
